In [1]:
from embedder import Embedder

model = Embedder()

q1 = "How does approximate nearest neighbor search work?"

2026-06-24 11:00:26.275092411 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


## Q1

In [2]:
v1 = model.encode(q1)
print(v1[0])

-0.02058203437252893


## Q2

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [7]:
c1 = [page['content'] for page in documents if page['filename'] == "02-vector-search/lessons/07-sqlitesearch-vector.md"][0]

In [8]:
v1.dot(model.encode(c1))

np.float64(0.36107026789538205)

## Q3

In [13]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

batch = [chunk['content'] for chunk in chunks]

X = model.encode_batch(batch)

scores = X.dot(v1)

In [21]:
import numpy as np
idx = np.argmax(scores)
idx, scores[idx], chunks[idx]['filename']

(np.int64(94),
 np.float64(0.6489016436447387),
 '02-vector-search/lessons/07-sqlitesearch-vector.md')

## Q4

In [22]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['filename'])
vindex.fit(X, chunks)

In [24]:
vindex.search(model.encode('What metric do we use to evaluate a search engine?'), num_results=1)

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

## Q5

In [32]:
from minsearch import Index
txtindex = Index(text_fields=['content'], keyword_fields=['filename'])
txtindex.fit(chunks)

In [ ]:
 txtindex.search("How do I store vectors in PostgreSQL?", num_results=5)

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin